# 06 Models, threshold, R/Y/G policy
RF classifier + IsolationForest + Autoencoder baseline; ROC/PR; cost-based threshold; R/Y/G policy and alarm volume/day. Output: predictions.parquet, THRESHOLD_POLICY.md, reports/figures/*.png

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split

ROOT = Path(".").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
DOCS = ROOT / "docs"
REPORTS = ROOT / "reports" / "figures"
DOCS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PROCESSED / "features_tabular.parquet")
X = df.drop(columns=["sample_id"], errors="ignore").values
y = np.zeros(len(X))  # placeholder labels
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr, y_tr)
iso = IsolationForest(random_state=42).fit(X_tr)
pred_rf = rf.predict_proba(X_te)[:, 1] if len(np.unique(y_tr)) > 1 else np.zeros(len(X_te))
pred_iso = iso.decision_function(X_te)
pred_df = pd.DataFrame({"rf_score": pred_rf, "iso_score": pred_iso})
pred_df.to_parquet(DATA_PROCESSED / "predictions.parquet", index=False)
(DOCS / "THRESHOLD_POLICY.md").write_text("# Threshold & R/Y/G Policy\n\nDefine Red/Yellow/Green bands and alarm volume/day here.\n")
print("Done.")